# ViT Position Encoding Analysis: SSDC, RPI, and Layer Windowed Ablation

This notebook reproduces the spatial structure analysis for two pretrained Vision Transformers, then runs a new mechanistic ablation experiment.

**Models**
- APE: `google/vit-base-patch16-224` (learned absolute position embeddings), via transformers.
- RoPE: `vit_base_patch16_rope_224.naver_in1k` (rotary position embeddings), via timm.

**Data**: ImageNet-1k validation, streamed from the Hugging Face Hub (gated, needs a token).

**What runs here**
1. SSDC across depth, clean and under Random Permutation at Inference (RPI), for both models.
2. Robustness (fragility) under ImageNet-C Gaussian blur.
3. The new experiment: zero ablating MLP or attention sublayers in early, middle, or late layer windows for the APE model. This asks whether the early SSDC peak is tied to the early MLPs specifically or just to the first MLPs the tokens meet, and whether removing attention only in later layers still flattens the later layer SSDC decay.
4. Extension: effective rank across depth and its collapse under MLP ablation (Dong et al., 2021).

Guidance paper: Mannes, *Positional Encodings Anchor Spatial Structure in Vision Transformers* (arXiv 2606.00124). The experiments here are our own and use pretrained ViT-Base models, not the from scratch ViT-S setup in that paper.

Use a GPU runtime (Runtime > Change runtime type > GPU).

## 0. Setup

In [ ]:
# Colab already ships torch, numpy, scipy, matplotlib, Pillow.
!pip -q install timm transformers datasets scikit-image huggingface_hub

In [ ]:
import os, sys

REPO = 'vit-sae-analysis'
if not os.path.exists(REPO):
    !git clone https://github.com/aravinds-kannappan/vit-sae-analysis.git

SRC = os.path.abspath(os.path.join(REPO, 'project_code', 'src'))
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print('src on path:', SRC)

# Optional: mount Drive to persist figures and results.
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# ImageNet-1k is gated. Paste a Hugging Face token with read access.
import os, getpass
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face token: ')
from huggingface_hub import login
login(os.environ['HF_TOKEN'], add_to_git_credential=False)

In [ ]:
import torch
import matplotlib.pyplot as plt
from experiments import common, reproduce_ssdc, reproduce_robustness, ablation_layerwise, effective_rank_probe

print('device:', 'cuda' if torch.cuda.is_available() else 'cpu (slow, switch to a GPU runtime)')

# Sample sizes. Raise for smoother curves, lower for a quick pass.
NUM_IMAGES = 1000
BATCH_SIZE = 128
ABLATION_IMAGES = 512   # the ablation sweep runs many conditions, keep it modest

# Stream ImageNet-1k validation once and reuse it (re-iterating restarts the stream).
dataset = common.load_imagenet(split='validation', streaming=True)

## 1. SSDC across depth, clean and under RPI

APE should show SSDC under RPI peaking early (blocks 2 to 3) then decaying with depth. RoPE should accumulate more gradually and peak later. Dashed lines are the reference curves stored in the repo.

In [ ]:
ssdc_results = {}
for kind in ['ape', 'rope']:
    print(f'=== {kind.upper()} ===')
    ssdc_results[kind] = reproduce_ssdc.run_model(kind, dataset, number_images=NUM_IMAGES, batch_size=BATCH_SIZE)
    print('clean:', [round(x, 3) for x in ssdc_results[kind]['clean']])
    print('rpi:  ', [round(x, 3) for x in ssdc_results[kind]['rpi']])
    reproduce_ssdc.plot_model(kind, ssdc_results[kind])
plt.show()

## 2. Robustness under Gaussian blur

Fragility = 1 - shifted / baseline accuracy under ImageNet-C Gaussian blur (severity 5). Reference: APE 0.326, RoPE 0.299 (RoPE is the more robust).

In [ ]:
rob = {}
for kind in ['ape', 'rope']:
    rob[kind] = reproduce_robustness.run_model(kind, dataset, corruption='Gaussian Blur', severity=5,
                                               number_images=NUM_IMAGES, batch_size=BATCH_SIZE)

print('\nmodel   baseline  shifted  fragility')
for kind in ['ape', 'rope']:
    r = rob[kind]
    print(f"{kind:5s}   {r['baseline_accuracy']:.3f}    {r['shifted_accuracy']:.3f}    {r['fragility']:.3f}")

## 3. New experiment: layer windowed MLP and attention ablation (APE)

Windows: early = [0,1,2,3], mid = [4,5,6,7], late = [8,9,10,11]. Primary metric is SSDC under RPI, since both the early peak and the later decay live in that curve for APE. This runs about twelve conditions, so give it a few minutes on a GPU. Three figures are produced: MLP ablation by window, keep MLPs in one window only, and attention ablation by window.

In [ ]:
abl = ablation_layerwise.run_all(dataset, model_kind='ape',
                                 number_images=ABLATION_IMAGES, batch_size=BATCH_SIZE,
                                 do_rank=False, do_plot=True)
plt.show()

### Reading the ablation result

- **MLP claim**: `mlp_zero_all` should crush SSDC under RPI toward zero at all depths. That is the *MLPs carry the index anchored recovery* signal.
- **Early MLP specificity**: compare `mlp_zero_early` with `mlp_zero_late`. If early ablation removes the early peak but late ablation leaves it, the peak is tied to the early MLP blocks.
- **First blocks encountered**: look at `mlp_keep_early`, `mlp_keep_mid`, `mlp_keep_late`. If the peak layer follows the surviving window (a later kept window gives a later peak), the peak tracks the first surviving MLPs rather than depth itself.
- **Attention and the later decay**: `attn_zero_all` should shrink the decay (SSDC stays elevated late). If `attn_zero_late` alone also shrinks it, the later layer decay is driven by attention in the late blocks specifically.

The printed summary table gives peak, peak_layer, delta, decay, final, and auc per condition to back these reads with numbers.

## 4. Extension: effective rank and rank collapse (Dong et al., 2021)

Attention without MLPs drives token representations toward rank one with depth. If MLP ablation collapses effective rank where it also collapses SSDC recovery, that links a representational capacity failure to the spatial structure failure, which is a mechanistic account rather than a correlation.

In [ ]:
rank_curves = effective_rank_probe.run('ape', dataset, number_images=ABLATION_IMAGES,
                                       batch_size=BATCH_SIZE, rpi=True)
common.plot_curves(rank_curves, title='APE effective rank across depth (under RPI)', ylabel='effective rank')
plt.show()

## Notes and next steps

- Every curve depends on the sampled images, so numbers vary slightly run to run. Raise `NUM_IMAGES` for tighter estimates.
- The ablated (trained without PE) and RPT conditions from the guidance paper need training from scratch and are out of scope for this pretrained study.
- Natural extensions: sweep the ablation window boundary layer by layer to find where the decay switches on, repeat the ablation study on RoPE to see whether its gradual accumulation responds differently, and connect ablation to the SAE features in `project_code/src/SAE`.